## Imports and configs

In [1]:
import duckdb
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
con = duckdb.connect("google_sheets_pipeline.duckdb")

## Inputs

In [2]:
stats = {'strength': 14, 
         'dexterity': 14,
         'intelligence': 40,
         'faith': 1,
         'arcane': 1}

In [3]:
weapon_of_choice = 'Cold Dagger'

In [4]:
weapon_upgrade = 6

## Calculation

In [5]:
df_weapon = con.sql(f"""SELECT
    rpw.id,
    epw.name,
    epw.attack_base_physics * rpw.physical_attack__v_double as upgraded_weapon_physical_attack,
    epw.attack_base_magic * rpw.magic_attack__v_double as upgraded_weapon_magic_attack, 
    epw.attack_base_fire * rpw.fire_attack__v_double as upgraded_weapon_fire_attack,
    epw.attack_base_thunder * rpw.lightning_attack__v_double as upgraded_weapon_lightning_attack,
    epw.attack_base_physics * rpw.physical_attack__v_double * (epw.correct_strength * rpw.str_scaling__v_double)/100 as physical_on_strength_scaling,
    epw.attack_base_physics * rpw.physical_attack__v_double * (epw.correct_agility * rpw.dex_scaling__v_double)/100 as physical_on_dexterity_scaling, 
    epw.attack_base_physics * rpw.physical_attack__v_double * (epw.correct_magic * rpw.int_scaling__v_double)/100 as physical_on_intelligence_scaling,
    epw.attack_base_physics * rpw.physical_attack__v_double * (epw.correct_faith * rpw.fai_scaling__v_double)/100 as physical_on_faith_scaling,
    epw.attack_base_physics * rpw.physical_attack__v_double * (epw.correct_luck * rpw.arc_scaling__v_double)/100 as physical_on_arcane_scaling,
    epw.attack_base_magic * rpw.magic_attack__v_double * (epw.correct_strength * rpw.str_scaling__v_double)/100 as magic_on_strength_scaling, 
    epw.attack_base_magic * rpw.magic_attack__v_double * (epw.correct_agility * rpw.dex_scaling__v_double)/100 as magic_on_dexterity_scaling, 
    epw.attack_base_magic * rpw.magic_attack__v_double * (epw.correct_magic * rpw.int_scaling__v_double)/100 as magic_on_intelligence_scaling,
    epw.attack_base_magic * rpw.magic_attack__v_double * (epw.correct_faith * rpw.fai_scaling__v_double)/100 as magic_on_faith_scaling,
    epw.attack_base_magic * rpw.magic_attack__v_double * (epw.correct_luck * rpw.arc_scaling__v_double)/100 as magic_on_arcane_scaling,
    epw.attack_base_fire * rpw.fire_attack__v_double * (epw.correct_strength * rpw.str_scaling__v_double)/100 as fire_on_strength_scaling,
    epw.attack_base_fire * rpw.fire_attack__v_double * (epw.correct_agility * rpw.dex_scaling__v_double)/100 as fire_on_dexterity_scaling, 
    epw.attack_base_fire * rpw.fire_attack__v_double * (epw.correct_magic * rpw.int_scaling__v_double)/100 as fire_on_intelligence_scaling,
    epw.attack_base_fire * rpw.fire_attack__v_double * (epw.correct_faith * rpw.fai_scaling__v_double)/100 as fire_on_faith_scaling,
    epw.attack_base_fire * rpw.fire_attack__v_double * (epw.correct_luck * rpw.arc_scaling__v_double)/100 as fire_on_arcane_scaling,
    epw.attack_base_thunder * rpw.lightning_attack__v_double * (epw.correct_strength * rpw.str_scaling__v_double)/100 as lightning_on_strength_scaling,
    epw.attack_base_thunder * rpw.lightning_attack__v_double * (epw.correct_agility * rpw.dex_scaling__v_double)/100 as lightning_on_dexterity_scaling, 
    epw.attack_base_thunder * rpw.lightning_attack__v_double * (epw.correct_magic * rpw.int_scaling__v_double)/100 as lightning_on_intelligence_scaling,
    epw.attack_base_thunder * rpw.lightning_attack__v_double * (epw.correct_faith * rpw.fai_scaling__v_double)/100 as lightning_on_faith_scaling,
    epw.attack_base_thunder * rpw.lightning_attack__v_double * (epw.correct_luck * rpw.arc_scaling__v_double)/100 as lightning_on_arcane_scaling
FROM
    google_sheets_pipeline.sample_google_sheet_data_20240610102202.equip_param_weapon epw
CROSS JOIN 
    google_sheets_pipeline.sample_google_sheet_data_20240610102202.reinforce_param_weapon rpw 
WHERE 1=1 
    AND epw.reinforce_type_id <= (rpw.id + 25)
    AND rpw.id - epw.reinforce_type_id <= 25
    AND epw.name = '{weapon_of_choice}' -- FILTRANDO NOME DA ARMA
    -- AND rpw.id = 905 -- FILTRANDO NÍVEL DA ARMA
""").df().loc[[weapon_upgrade]]
df_weapon

,id,name,upgraded_weapon_physical_attack,upgraded_weapon_magic_attack,upgraded_weapon_fire_attack,upgraded_weapon_lightning_attack,physical_on_strength_scaling,physical_on_dexterity_scaling,physical_on_intelligence_scaling,physical_on_faith_scaling,physical_on_arcane_scaling,magic_on_strength_scaling,magic_on_dexterity_scaling,magic_on_intelligence_scaling,magic_on_faith_scaling,magic_on_arcane_scaling,fire_on_strength_scaling,fire_on_dexterity_scaling,fire_on_intelligence_scaling,fire_on_faith_scaling,fire_on_arcane_scaling,lightning_on_strength_scaling,lightning_on_dexterity_scaling,lightning_on_intelligence_scaling,lightning_on_faith_scaling,lightning_on_arcane_scaling
6,906,Cold Dagger,76.538,64.68,0.0,0.0,30.844814,57.709652,51.739688,0.0,0.0,26.06604,48.76872,43.72368,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [6]:
df = con.sql(f"""WITH calc_correct_union AS (SELECT
	'physical' AS damage_type,
	*
FROM
	google_sheets_pipeline.sample_google_sheet_data_20240610102202.calc_correct_graph ccg
JOIN google_sheets_pipeline.sample_google_sheet_data_20240610102202.calc_correct_graph_id ccgi ON ccg.id = ccgi.calc_correct_graph_id_physicalx 

UNION ALL 

SELECT
	'magic' AS damage_type,
	*
FROM
	google_sheets_pipeline.sample_google_sheet_data_20240610102202.calc_correct_graph ccg
JOIN google_sheets_pipeline.sample_google_sheet_data_20240610102202.calc_correct_graph_id ccgi ON ccg.id = ccgi.calc_correct_graph_id_magicx  

UNION ALL 

SELECT
	'fire' AS damage_type,
	*
FROM
	google_sheets_pipeline.sample_google_sheet_data_20240610102202.calc_correct_graph ccg
JOIN google_sheets_pipeline.sample_google_sheet_data_20240610102202.calc_correct_graph_id ccgi ON ccg.id = ccgi.calc_correct_graph_id_firex 

UNION ALL 

SELECT
	'lightning' AS damage_type,
	*
FROM
	google_sheets_pipeline.sample_google_sheet_data_20240610102202.calc_correct_graph ccg
JOIN google_sheets_pipeline.sample_google_sheet_data_20240610102202.calc_correct_graph_id ccgi ON ccg.id = ccgi.calc_correct_graph_id_lightningx 

UNION ALL 

SELECT
	'holy' AS damage_type,
	*
FROM
	google_sheets_pipeline.sample_google_sheet_data_20240610102202.calc_correct_graph ccg
JOIN google_sheets_pipeline.sample_google_sheet_data_20240610102202.calc_correct_graph_id ccgi ON ccg.id = ccgi.calc_correct_graph_id_holyx)

SELECT * FROM calc_correct_union WHERE name = '{weapon_of_choice}' -- FILTRANDO NOME DA ARMA
""").df()

df

,damage_type,id,used_for,stat_0,stat_1,stat_2,stat_3,stat_4,grow_0,grow_1,grow_2,grow_3,grow_4,exponent_0,exponent_1,exponent_2,exponent_3,exponent_4,_dlt_load_id,_dlt_id,exponent_2__v_double,exponent_3__v_double,grow_1__v_double,grow_2__v_double,grow_3__v_double,grow_4__v_double,id_1,name,calc_correct_graph_id_physicalx,calc_correct_graph_id_magicx,calc_correct_graph_id_firex,calc_correct_graph_id_lightningx,calc_correct_graph_id_holyx,_dlt_load_id_1,_dlt_id_1
0,physical,0,Weapon Scaling (Default),1,18,60,80,150,0,25,75,90,110,1.2,-1.2,1,1,1,1718058129.4026427,FP2GSngqWVNh5w,NaN,NaN,NaN,NaN,NaN,NaN,1000900,Cold Dagger,0,0,0,0,0,1718058129.4026427,QfJ963vwIBkWOg
1,magic,0,Weapon Scaling (Default),1,18,60,80,150,0,25,75,90,110,1.2,-1.2,1,1,1,1718058129.4026427,FP2GSngqWVNh5w,NaN,NaN,NaN,NaN,NaN,NaN,1000900,Cold Dagger,0,0,0,0,0,1718058129.4026427,QfJ963vwIBkWOg
2,fire,0,Weapon Scaling (Default),1,18,60,80,150,0,25,75,90,110,1.2,-1.2,1,1,1,1718058129.4026427,FP2GSngqWVNh5w,NaN,NaN,NaN,NaN,NaN,NaN,1000900,Cold Dagger,0,0,0,0,0,1718058129.4026427,QfJ963vwIBkWOg
3,lightning,0,Weapon Scaling (Default),1,18,60,80,150,0,25,75,90,110,1.2,-1.2,1,1,1,1718058129.4026427,FP2GSngqWVNh5w,NaN,NaN,NaN,NaN,NaN,NaN,1000900,Cold Dagger,0,0,0,0,0,1718058129.4026427,QfJ963vwIBkWOg
4,holy,0,Weapon Scaling (Default),1,18,60,80,150,0,25,75,90,110,1.2,-1.2,1,1,1,1718058129.4026427,FP2GSngqWVNh5w,NaN,NaN,NaN,NaN,NaN,NaN,1000900,Cold Dagger,0,0,0,0,0,1718058129.4026427,QfJ963vwIBkWOg


In [7]:
df_bin = con.sql(f"""
SELECT
	aecp.*
FROM google_sheets_pipeline.sample_google_sheet_data_20240610102202.equip_param_weapon epw
JOIN sample_google_sheet_data_20240610102202.attack_element_correct_param aecp ON aecp.row_id = epw.attack_element_correct_id

WHERE epw.name = '{weapon_of_choice}' -- FILTRANDO NOME DA ARMA
""").df()
df_bin

,row_id,physical_scaling_str,physical_scaling_dex,physical_scaling_int,physical_scaling_fai,physical_scaling_arc,magic_scaling_str,magic_scaling_dex,magic_scaling_int,magic_scaling_fai,magic_scaling_arc,fire_scaling_str,fire_scaling_dex,fire_scaling_int,fire_scaling_fai,fire_scaling_arc,lightning_scaling_str,lightning_scaling_dex,lightning_scaling_int,lightning_scaling_fai,lightning_scaling_arc,holy_scaling_str,holy_scaling_dex,holy_scaling_int,holy_scaling_fai,holy_scaling_arc,_dlt_load_id,_dlt_id
0,10000,1,1,0,0,0,0,0,1,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,1718058129.4026427,5li5Lg3y3ckS8g


In [8]:
def GetCalcCorrectFormulaInputs(stats, df):
    results = {}
    for stat in stats:
        input = stats[stat]
        if input <= df['stat_1'].iloc[0]:
            posmin = 0
            posmax = 1
        elif input <= df['stat_2'].iloc[0]:
            posmin = 1
            posmax = 2
        elif input <= df['stat_3'].iloc[0]:
            posmin = 2
            posmax = 3
        elif input <= df['stat_4'].iloc[0]:
            posmin = 3
            posmax = 4
    
        statmin = df[f'stat_{posmin}'].iloc[0]
        statmax = df[f'stat_{posmax}'].iloc[0]
        growmin = df[f'grow_{posmin}'].iloc[0]
        growmax = df[f'grow_{posmax}'].iloc[0]
        exponentmin = df[f'exponent_{posmin}'].iloc[0]

        results[f'{stat}'] = {
            'input': input, 
            'statmin': statmin, 
            'statmax': statmax, 
            'growmin': growmin, 
            'growmax': growmax, 
            'exponentmin': exponentmin
        }
    return results

In [9]:
def CalcCorrectFormula(stat_payload):

    input = stat_payload['input']
    statmin = stat_payload['statmin']
    statmax = stat_payload['statmax']
    growmin = stat_payload['growmin']
    growmax = stat_payload['growmax']
    exponentmin = stat_payload['exponentmin']
    
    ratio = (input - statmin) / (statmax - statmin)
    
    if exponentmin > 0:
        growth = ratio ** exponentmin
        
    if exponentmin < 0:
        growth = 1 - (1 - (ratio))**abs(exponentmin)

    result = growmin + ((growmax - growmin) * growth)
    
    return result

def GetDamageByStats(df, stats):
    results = {}
    for i in df['damage_type']:
        for stat in stats:
            if df_bin[f'{i}_scaling_{stat[:3]}'].iloc[0] == 1:
                formula_inputs = GetCalcCorrectFormulaInputs(stats, df.loc[df['damage_type'] == f'{i}'])
                formula_output = CalcCorrectFormula(formula_inputs[stat])/100
                if formula_output > 0:
                    results[f'stat_multiplier_{i}_on_{stat}'] = formula_output
    return results

In [10]:
def FinalScalingCalc(df, df_weapon):
    results = {}
    for i in df:
        join = i.replace('stat_multiplier_', '')
        for j in df_weapon.columns:
            if join == j.replace('_scaling', ''):
                result = df_weapon[f'{join}_scaling'].iloc[0]  *  df[i]
                if result > 0:
                    results[f'scaled_{i.split("_")[2]}_attack_on_{i.split("_")[4]}'] = result
    
    return results

In [11]:
def TotalAttackPower(dict):
    physical_attack = df_weapon['upgraded_weapon_physical_attack'].iloc[0]
    magic_attack = df_weapon['upgraded_weapon_magic_attack'].iloc[0]
    fire_attack = df_weapon['upgraded_weapon_fire_attack'].iloc[0]
    lightning_attack = df_weapon['upgraded_weapon_lightning_attack'].iloc[0]
    
    for i in dict:
        stat_name = i.split('_')[1]
        if stat_name == 'physical':
              physical_attack += dict[i]
        if stat_name == 'magic':
              magic_attack += dict[i]
        if stat_name == 'fire':
              fire_attack += dict[i]
        if stat_name == 'lightning':
              lightning_attack += dict[i]
    
    total_weapon_attack_power =  physical_attack + magic_attack + fire_attack + lightning_attack
    return total_weapon_attack_power

TotalAttackPower(FinalScalingCalc(GetDamageByStats(df, stats), df_weapon))

181.08113571434228